In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
model_name = "MLRS/BERTu"

params = {
    "learning_rate": 2e-5,
    "batch_size": 16,
    "weight_decay": 0.1,
    "max_length": 128
}

seeds = [1,2,3,4,5]

In [ ]:
# Load the Maltese Semantic Analysis Dataset
dataset = load_dataset("DGurgurov/maltese_sa")

train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])
test_df = pd.DataFrame(dataset["test"])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=params["max_length"]
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

In [ ]:
results = []

for seed in seeds:

    print(f"\nTraining with seed {seed}\n")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    output_dir = f"./seeds/bert_seed_{seed}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=params["learning_rate"],
        per_device_train_batch_size=params["batch_size"],
        per_device_eval_batch_size=params["batch_size"],
        num_train_epochs=200,
        weight_decay=params["weight_decay"],
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=seed,
        report_to="none",
        disable_tqdm=False,
        save_total_limit=1,
        #warmup_ratio=0.1,
        #lr_scheduler_type="linear",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20)]
    )

    trainer.train()

    test_output = trainer.predict(tokenized_dataset["test"])
    test_metrics = compute_metrics((test_output.predictions, test_output.label_ids))

    results.append({
        "seed": seed,
        "test_f1": test_metrics["macro_f1"]
    })

    print("Test F1:", test_metrics["macro_f1"])

In [ ]:
results_df = pd.DataFrame(results)

mean_f1 = results_df["test_f1"].mean()
std_f1 = results_df["test_f1"].std()

print(results_df)
print(f"\nFinal Result: F1 = {mean_f1:.3f} ± {std_f1:.3f}")